# NeoOLAF DocRED Run and Controlled Evaluation KG Export

This notebook runs the existing NeoOLAF benchmark runner without modifying
NeoOLAF source code. It then projects each native `kg_local.json` and
`kg_inferred.json` into evaluation files constrained by a fixed relation
vocabulary.

NeoOLAF may still discover and add new relations to its generated ontology.
Those relations remain in the native artifacts but are excluded from the
evaluation KG unless they are permitted by the selected reference
ontology/list mode.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path:
    # Support launching from the repository root or examples/RAGTreeDatasets.
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "experiments/methods/run_neoolaf.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find experiments/methods/run_neoolaf.py. "
        "Launch this notebook from inside the NeoOLAF/RAGTree repository."
    )


PROJECT_ROOT = find_project_root()
RUNNER_PATH = PROJECT_ROOT / "experiments/methods/run_neoolaf.py"
PROJECTOR_PATH = PROJECT_ROOT / "experiments/methods/evaluation_kg_projection.py"
OFFLINE_RUNTIME_DIR = PROJECT_ROOT / "experiments/methods/neoolaf_offline_runtime"

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUNNER_PATH={RUNNER_PATH}")
print(f"PROJECTOR_PATH={PROJECTOR_PATH}")
print(f"OFFLINE_RUNTIME_DIR={OFFLINE_RUNTIME_DIR}")

## Configuration

`RELATION_FILTER_MODE` accepts:

- `list-only`, recommended for DocRED
- `ontology-only`
- `union`
- `intersection`

`REFERENCE_ONTOLOGY_PATH` is used only by the evaluation projector. It
must be a fixed dataset/reference ontology, not the ontology generated by
the current NeoOLAF run.

In [ ]:
# Dataset and NeoOLAF inputs.
DATASET_JSONL = PROJECT_ROOT / "data/RAGTreeDatasets/docred/dev.jsonl"
NEOOLAF_ONTOLOGY_PATH = PROJECT_ROOT / "data/ontology/redocred_ontology.ttl"
USER_GUIDANCE_PATH = (
    PROJECT_ROOT / "examples/RAGTreeDatasets/configs/guidance_docred.json"
)

# Benchmark outputs.
RUNS_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets/runs"
OUTPUT_JSONL = RUNS_DIR / "neoolaf_docred_predictions.docred_constrained.canonical.jsonl"
ARTIFACTS_ROOT = RUNS_DIR / "neoolaf_docred_artifacts_docred_constrained"
SUMMARY_OUTPUT = RUNS_DIR / "neoolaf_docred_run_summary.json"
ERROR_LOG_JSONL = RUNS_DIR / "neoolaf_docred_errors.jsonl"
ARTIFACT_INDEX_JSONL = RUNS_DIR / "neoolaf_docred_evaluation_artifact_index.jsonl"

# OpenRouter configuration.
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY", "")
MODEL_NAME = "openai/gpt-oss-20b"

# Smoke test first, then set MAX_DOCS=None for the full split.
RUN_NEOOLAF = True
MAX_DOCS = 5
DOCUMENT_WORKERS = 2
DISABLE_WIKIPEDIA = True

# Controlled evaluation relation vocabulary.
RELATION_FILTER_MODE = "list-only"
REFERENCE_ONTOLOGY_PATH = NEOOLAF_ONTOLOGY_PATH
RELATION_VOCAB_PATH = None

# This extracts only the set of relation IDs/labels from the full dataset.
# It never copies gold subject-object pairs into predictions.
SOURCE_RELATION_VOCAB_PATH = DATASET_JSONL

# Add runner-only flags here if your local wrapper supports them, for
# example offline/Wikipedia controls. NeoOLAF source is not changed.
EXTRA_RUNNER_ARGS: list[str] = []

RUNS_DIR.mkdir(parents=True, exist_ok=True)

if RUN_NEOOLAF and not OPENROUTER_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY before running NeoOLAF.")

## Run NeoOLAF

In [ ]:
command = [
    sys.executable,
    str(RUNNER_PATH),
    "--dataset-jsonl-path", str(DATASET_JSONL),
    "--ontology-path", str(NEOOLAF_ONTOLOGY_PATH),
    "--output-jsonl-path", str(OUTPUT_JSONL),
    "--backend-name", "openrouter",
    "--host", OPENROUTER_HOST,
    "--api-key", OPENROUTER_KEY,
    "--model-name", MODEL_NAME,
    "--type-filter", "dev",
    "--user-guidance-path", str(USER_GUIDANCE_PATH),
    "--few-shot-from-dataset",
    "--few-shot-source-type", "dev",
    "--few-shot-k", "1",
    "--output-format", "canonical",
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--chunk-size", "10000000",
    "--chunk-overlap", "0",
    "--max-chunks", "1",
    "--max-expressions", "20",
    "--max-relation-mentions", "20",
    "--max-workers", "1",
    "--document-workers", str(DOCUMENT_WORKERS),
    "--max-tokens", "8192",
    "--openrouter-reasoning-effort", "minimal",
    "--openrouter-exclude-reasoning",
    "--error-log-jsonl-path", str(ERROR_LOG_JSONL),
    "--summary-output-path", str(SUMMARY_OUTPUT),
    "--no-checkpoints",
    "--no-chunk-checkpoints",
    "--no-resume",
]

if MAX_DOCS is not None:
    command.extend(["--max-docs", str(MAX_DOCS)])
command.extend(EXTRA_RUNNER_ARGS)

print("NeoOLAF command:")
print(shlex.join(command))

if RUN_NEOOLAF:
    runner_env = os.environ.copy()
    if DISABLE_WIKIPEDIA:
        if not (OFFLINE_RUNTIME_DIR / "sitecustomize.py").is_file():
            raise FileNotFoundError(
                f"Offline runtime policy not found: {OFFLINE_RUNTIME_DIR}"
            )
        current_pythonpath = runner_env.get("PYTHONPATH", "")
        runner_env["PYTHONPATH"] = os.pathsep.join(
            value
            for value in (str(OFFLINE_RUNTIME_DIR), current_pythonpath)
            if value
        )
        runner_env["NEOOLAF_DISABLE_WIKIPEDIA"] = "1"
        print("[NeoOLAF benchmark] DISABLE_WIKIPEDIA=True")

    subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env=runner_env,
        check=True,
    )
else:
    print("RUN_NEOOLAF=False, using the existing prediction JSONL and artifacts.")

## Create Controlled Evaluation KGs

This stage reads native NeoOLAF artifacts only. The source document
supplies canonical entities, while the configured relation vocabulary
controls which predicates may appear in evaluation.

In [ ]:
def iter_jsonl(path: Path):
    # Stream records to avoid loading a full benchmark file into memory.
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)


def resolve_artifact_dir(raw_path: str) -> Path:
    # Runner outputs may store absolute paths or paths relative to the repo.
    path = Path(raw_path)
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def find_native_kg_exports(artifact_dir: Path) -> tuple[Path, Path]:
    # Prefer local/inferred exports that share the same directory.
    local_candidates = sorted(
        artifact_dir.rglob("kg_local.json"),
        key=lambda path: ("data/exports" not in path.as_posix(), len(path.parts)),
    )
    for local_path in local_candidates:
        inferred_path = local_path.with_name("kg_inferred.json")
        if inferred_path.is_file():
            return local_path, inferred_path
    raise FileNotFoundError(
        f"No kg_local.json + kg_inferred.json pair found under {artifact_dir}"
    )


def projection_command(
    document_id: str,
    kg_local: Path,
    kg_inferred: Path,
    output_dir: Path,
) -> list[str]:
    # Build the independent post-processing command.
    command = [
        sys.executable,
        str(PROJECTOR_PATH),
        "--kg-local-json", str(kg_local),
        "--kg-inferred-json", str(kg_inferred),
        "--source-doc-json", str(DATASET_JSONL),
        "--document-id", document_id,
        "--relation-filter-mode", RELATION_FILTER_MODE,
        "--output-dir", str(output_dir),
    ]
    if REFERENCE_ONTOLOGY_PATH is not None:
        command.extend(["--ontology-path", str(REFERENCE_ONTOLOGY_PATH)])
    if RELATION_VOCAB_PATH is not None:
        command.extend(["--relation-vocab-json", str(RELATION_VOCAB_PATH)])
    if SOURCE_RELATION_VOCAB_PATH is not None:
        command.extend(
            ["--source-relation-vocab-json", str(SOURCE_RELATION_VOCAB_PATH)]
        )
    return command

In [ ]:
if not OUTPUT_JSONL.is_file():
    raise FileNotFoundError(f"Prediction file not found: {OUTPUT_JSONL}")
if not PROJECTOR_PATH.is_file():
    raise FileNotFoundError(f"Evaluation projector not found: {PROJECTOR_PATH}")

artifact_index = []
projection_errors = []

for prediction_record in iter_jsonl(OUTPUT_JSONL):
    if not prediction_record.get("parsed_ok", False):
        continue

    document_id = str(prediction_record["document_id"])
    artifact_dir = resolve_artifact_dir(prediction_record["artifact_dir"])

    try:
        kg_local, kg_inferred = find_native_kg_exports(artifact_dir)
        exports_dir = kg_local.parent
        command = projection_command(
            document_id=document_id,
            kg_local=kg_local,
            kg_inferred=kg_inferred,
            output_dir=exports_dir,
        )
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)

        index_record = {
            "document_id": document_id,
            "artifact_dir": str(artifact_dir),
            "exports_dir": str(exports_dir),
            "kg_local": str(kg_local),
            "kg_inferred": str(kg_inferred),
            "evaluation_kg_local": str(exports_dir / "evaluation_kg_local.json"),
            "evaluation_kg_inferred": str(exports_dir / "evaluation_kg_inferred.json"),
            "evaluation_kg_combined": str(exports_dir / "evaluation_kg_combined.json"),
            "evaluation_projection_report": str(
                exports_dir / "evaluation_projection_report.json"
            ),
            "relation_filter_mode": RELATION_FILTER_MODE,
        }
        artifact_index.append(index_record)

        print(f"[NeoOLAF benchmark][exports] document_id={document_id}")
        for key, value in index_record.items():
            if key not in {"document_id", "relation_filter_mode"}:
                print(f"[NeoOLAF benchmark][exports] {key}={value}")
    except Exception as error:
        projection_errors.append(
            {
                "document_id": document_id,
                "artifact_dir": str(artifact_dir),
                "error_type": type(error).__name__,
                "error": str(error),
            }
        )
        print(
            f"[NeoOLAF benchmark][evaluation-error] document_id={document_id} "
            f"{type(error).__name__}: {error}"
        )

with ARTIFACT_INDEX_JSONL.open("w", encoding="utf-8") as handle:
    for record in artifact_index:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"[NeoOLAF benchmark][exports] artifact_index={ARTIFACT_INDEX_JSONL}")
print(
    f"[NeoOLAF benchmark][exports] projected={len(artifact_index)} "
    f"errors={len(projection_errors)}"
)

if projection_errors:
    error_path = RUNS_DIR / "neoolaf_docred_evaluation_projection_errors.json"
    error_path.write_text(
        json.dumps(projection_errors, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"[NeoOLAF benchmark][exports] projection_errors={error_path}")

## Projection Summary

In [ ]:
summary_rows = []
for record in artifact_index:
    report_path = Path(record["evaluation_projection_report"])
    report = json.loads(report_path.read_text(encoding="utf-8"))
    row = {
        "document_id": record["document_id"],
        "mode": report["relation_filter_mode"],
        "entities": report["entity_count"],
        "allowed_relations": report["allowed_relation_count"],
    }
    for source_report in report["reports"]:
        source = source_report["source_name"]
        row[f"{source}_entities"] = source_report["projected_entities"]
        row[f"{source}_relations"] = source_report["projected_relations"]
        row[f"{source}_accepted"] = source_report["accepted_triples"]
        row[f"{source}_rejected"] = source_report["rejected_triples"]
    summary_rows.append(row)

print(json.dumps(summary_rows, ensure_ascii=False, indent=2))